In [53]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field

In [54]:
@dataclass
class VisionConfig:
  # image
  image_size: int = 224
  patch_size: int = 16
  image_channels: int = 3

  # vision transformer
  embedding_size: int = 768
  n_layers: int = 12
  n_heads: int = 12
  mlp_ratio: float = 4.0 # [hidden size of the MLP block]
  dropout: float = 0.0

@dataclass
class TextConfig:
  # tokenizer
  vocab_size: int = 49_408
  block_size: int = 77

  # text transformer
  embedding_size: int = 512
  n_layers: int = 12
  n_heads: int = 8
  mlp_ratio: float = 4.0
  dropout: float = 0.0

@dataclass
class CLIPConfig:
  vision: VisionConfig = field(default_factory=VisionConfig)
  text: TextConfig = field(default_factory=TextConfig)

  projection_dim: int = 512 # [shared embedding space]
  temperature: float = 0.7 # [initial temperature for the learnable logit scale]

In [55]:
@dataclass
class Config:
    image_size: int = 224
    patch_size: int = 16
    image_channels: int = 3

    vision_embeding_dim: int = 768
    vision_n_layers: int = 12
    vision_n_heads: int = 12

    vocab_size: int = 49_408
    context_length: int = 77

    text_embeding_dim: int = 512
    text_n_layers: int = 12
    text_n_heads: int = 8

    projection_dim: int = 512
    dropout: float = 0.0
    temperature: float = 0.07

In [56]:
class PatchEmbedding(nn.Module):
  def __init__(self, config: VisionConfig):
    super().__init__()
    # [convert an image into a sequence of vectors, where each vector represents one image patch]
    assert config.image_size % config.patch_size == 0, (
    f"image_size ({config.image_size}) must be divisible by "
    f"patch_size ({config.patch_size})"
    )

    self.num_patches = (config.image_size // config.patch_size) ** 2 # (224 * 224) // (16 * 16) = 196 patches
    self.projection = nn.Conv2d(
        in_channels = config.image_channels,
        out_channels = config.embedding_size,
        kernel_size = config.patch_size,
        stride = config.patch_size
    )

  def forward(self, images):
    # images (B, C, H, W)
    patches = self.projection(images) # (B, embedding_size, H/patch_size, W/patch_size)
    patches = patches.flatten(2) # (B, embedding_size, num_patches)
    patches = patches.transpose(1, 2) # (B, num_patches, embedding_size)
    return patches

In [57]:
class TransformerBlock(nn.Module):
  def __init__(self, config): # [config: VisionConfig or TextConfig]
    super().__init__()
    self.layer_norm_1 = nn.LayerNorm(config.embedding_size)
    self.mha = nn.MultiheadAttention(embed_dim=config.embedding_size, num_heads=config.n_heads, dropout=config.dropout, batch_first=True)
    self.layer_norm_2 = nn.LayerNorm(config.embedding_size)
    hidden_size = int(config.embedding_size * config.mlp_ratio)
    self.mlp = nn.Sequential(
        nn.Linear(config.embedding_size, hidden_size),
        nn.Dropout(config.dropout),
        nn.GELU(),
        nn.Linear(hidden_size, config.embedding_size),
        nn.Dropout(config.dropout)
    )

  def forward(self, x, attention_mask=None):
    # x: (B, T, C)
    x = self.layer_norm_1(x) # (B, T, C)
    attn_output, _ = self.mha(x, x, x, attn_mask=attention_mask, need_weights=False) # (B, T, C)
    x = x + attn_output # (B, T, C)
    x = x + self.mlp(self.layer_norm_2(x)) # (B, T, C)
    return x

In [58]:
class VisionEncoder(nn.Module):
  def __init__(self, config: VisionConfig):
    super().__init__()
    self.patch_embedding = PatchEmbedding(config)
    num_patches = self.patch_embedding.num_patches

    # [learnable [CLS]-style token that will be prepended to the patch sequence]
    self.class_token = nn.Parameter(torch.randn(1, 1, config.embedding_size) * 0.02) # (1 shared batch, 1 cls token, embedding_size)

    # [learnable position embedding: patch tokens + 1 class token]
    self.position_embedding = nn.Parameter(torch.randn(1, num_patches+1, config.embedding_size) * 0.02)

    self.dropout = nn.Dropout(config.dropout)
    self.transformer_blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
    self.layer_norm = nn.LayerNorm(config.embedding_size)

  def forward(self, images):
    # images: (B, C, H, W)
    x = self.patch_embedding(images) # (B, num_patches, embedding_size)
    batch_size = x.shape[0]

    class_token = self.class_token.expand(batch_size, -1, -1) # (1, 1, embedding_size) -> (B, 1, embedding_size)
    x = torch.cat([class_token, x], dim=1) # (B, num_patches + 1, embedding_size)

    x = x + self.position_embedding # (B, num_patches + 1, embedding_size)
    x = self.dropout(x)

    for block in self.transformer_blocks:
      x = block(x) # (B, num_patches + 1, embedding_size)
    x = self.layer_norm(x) # (B, num_patches + 1, embedding_size)

    # [take the class token as the image representation]
    image_features = x[:, 0] # (B, embedding_size)
    return image_features

In [59]:
class TextEncoder(nn.Module):
  def __init__(self, config: TextConfig):
    super().__init__()
    self.block_size = config.block_size
    self.token_embedding = nn.Embedding(config.vocab_size, config.embedding_size) # (V, C)
    self.position_embedding = nn.Parameter(torch.randn(1, self.block_size, config.embedding_size) * 0.02) # [trainable] (B, T, C)
    self.transformer_blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
    self.layer_norm = nn.LayerNorm(config.embedding_size)

    # [causal mask so each token can only attend to itself and earlier tokens]
    causal_mask = torch.full((self.block_size, self.block_size), float("-inf")) # (T, T)
    causal_mask = torch.triu(causal_mask, diagonal=1) # (T, T)
    self.register_buffer("causal_mask", causal_mask, persistent=False)

  def forward(self, token_ids, eos_positions):
    # token_ids: (B, <=T)
    # eos_positions: (B,) [index of the end-of-text token in each sequence]

    seq_len = token_ids.shape[1] #(<=T,)

    if seq_len > self.block_size:
      raise ValueError(
          f"Sequence length {seq_len} exceeds "
          f"context length {self.block_size}"
      )

    x = self.token_embedding(token_ids) # (B, <=T, C)
    x = x + self.position_embedding[:, :seq_len] # (B, <=T, C)

    mask = self.causal_mask[:seq_len, :seq_len] # (<=T, <=T)

    for block in self.transformer_blocks:
      x = block(x, attention_mask=mask) # (B, <=T, C)

    x = self.layer_norm(x) # (B, <=T, C)
    batch_indices = torch.arange(x.shape[0], device=x.device) # (B,)

    # the EOS token can attend to every earlier token, so its final vector can
    # serve as a representation of the entire sentence.
    text_features = x[batch_indices, eos_positions] # (B, C)
    return text_features

In [60]:
class CLIP(nn.Module):
  def __init__(self, config: CLIPConfig):
    super().__init__()

    self.image_encoder = VisionEncoder(config.vision)
    self.text_encoder = TextEncoder(config.text)

    # [project both modalities into a shared embedding space]
    self.image_projection = nn.Linear(config.vision.embedding_size, config.projection_dim, bias=False)
    self.text_projection = nn.Linear(config.text.embedding_size, config.projection_dim, bias=False)

    # [learnable temperature]
    self.logit_scale = nn.Parameter(torch.log(torch.tensor(1.0 / config.temperature)))

  def encode_image(self, images):
    # images: (B, C, H, W)
    image_features = self.image_encoder(images) # (B, vision.embedding_size)
    image_embeddings = self.image_projection(image_features) # (B, projection_dim)
    image_embeddings = F.normalize(image_embeddings, p=2, dim=-1) # (B, projection_dim) [L2-normalized]
    return image_embeddings

  def encode_text(self, token_ids, eos_positions):
    # token_ids: (B, T), eos_positions: (B,)
    text_features = self.text_encoder(token_ids, eos_positions) # (B, text.embedding_size)
    text_embeddings = self.text_projection(text_features) # (B, projection_dim)
    text_embeddings = F.normalize(text_embeddings, p=2, dim=-1) # (B, projection_dim) [L2-normalized]
    return text_embeddings

  def forward(self, images, token_ids, eos_positions):
    image_embeddings = self.encode_image(images) # (B, projection_dim)
    text_embeddings = self.encode_text(token_ids, eos_positions) # (B, projection_dim)

    # [convert back to the actual scale]
    scale = torch.exp(self.logit_scale).clamp(max=100)

    # cosine similarity
    logits_per_image = (scale * image_embeddings @ text_embeddings.T) # (B, projection_dim) @ (projection_dim, B) => (B, B)
    logits_per_text = logits_per_image.T # (B, B)
    return logits_per_image, logits_per_text

In [61]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = CLIPConfig()
model = CLIP(config).to(device)

In [62]:
# [The model should make the diagonal similarity values large:
# CLIP calculates loss in both directions:
#   image-to-text: given an image, retrieve its caption
#   text-to-image: given a caption, retrieve its image]

def clip_loss(logits_per_image, logits_per_text):
  # logits_per_image, logits_per_text: (B, B)
  batch_size = logits_per_image.shape[0]
  # [correct pair is on the diagonal]
  targets = torch.arange(batch_size, device=device) # (B,)

  image_to_text_loss = F.cross_entropy(logits_per_image, targets)
  text_to_image_loss = F.cross_entropy(logits_per_text, targets)
  return (image_to_text_loss + text_to_image_loss) / 2

In [63]:
batch_size = 8
sequence_length = 32

images = torch.randn(
    batch_size,
    config.vision.image_channels,
    config.vision.image_size,
    config.vision.image_size,
    device=device,
)  # (B, C, H, W)

token_ids = torch.randint(
    low=0,
    high=config.text.vocab_size,
    size=(batch_size, sequence_length),
    device=device,
)  # (B, T)

eos_positions = torch.full(
    (batch_size,),
    sequence_length - 1,
    dtype=torch.long,
    device=device,
)  # (B,)

In [64]:
logits_per_image, logits_per_text = model(
    images,
    token_ids,
    eos_positions,
)

loss = clip_loss(
    logits_per_image,
    logits_per_text,
)

print("Image-to-text logits:", logits_per_image.shape)  # (B, B)
print("Text-to-image logits:", logits_per_text.shape)   # (B, B)
print("Loss:", loss.item())

Image-to-text logits: torch.Size([8, 8])
Text-to-image logits: torch.Size([8, 8])
Loss: 2.077378273010254
